# Original dataset [AI config dataset](https://se-uhd.de/ai-config-dataset/)

In [2]:
import re
import pandas as pd
import numpy as np
from helper import load_jsonl_to_df
from sklearn.metrics import cohen_kappa_score

In [24]:
df = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")

In [28]:
# df['repo_name'].nunique()
df.loc[df['pr_count_after']>=20, "num_context_files"].sum()

np.int64(1567)

In [3]:
df = pd.merge(pd.read_csv("../data/cv_fold_metrics.csv"), pd.read_csv("../data/final_model_dataset.csv"), on="repo", how="left")
# df = df.sort_values(by=["recall"])
df = pd.merge(
    df,
    pd.read_csv("../data/pre_acf_model_features.csv")[['repo', 'description']],
    on="repo",
    how="left"
)

In [2]:
# Original dataset
configured_repos = pd.read_csv("../data/configured_repos.csv")
configured_repos = configured_repos.rename(columns={'repo_name': 'repo'})
context_df = pd.read_csv("../data/context_files.csv")
skills = pd.read_csv("../data/skills.csv")
subagents = pd.read_csv("../data/subagents.csv") # subagent definitions
commands = pd.read_csv("../data/commands.csv") # custom commands
rules = pd.read_csv("../data/rules.csv") # rule files
settings = pd.read_csv("../data/settings.csv") # settings files
hooks = pd.read_csv("../data/hooks.csv") # hook configurations
mcp = pd.read_csv("../data/mcp.csv") # MCP configurations

# Our data
pr_df = pd.read_csv("../data/pull_request_combined.csv")
pr_five_df = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")
pr_feat_df = pd.read_csv("../data/prs_features.csv")
repo_df = pd.read_csv("../data/repository_combined.csv")
repo_df = repo_df.rename(columns={'id': 'repo_id'})

pr_df = pr_df.merge(
    repo_df[['repo_id', 'full_name']],
    on='repo_id',
    how='left'
).rename(columns={'full_name': 'repo'})

# Add the number of agent-authored PRs to the original repo dataframe
configured_repos = configured_repos.merge(
    pr_feat_df.groupby("repo").size().reset_index(name="agent_prs"),
    on="repo", how='left'
)

# 1. Convert string columns to datetime
# Using utc=True is safer if your timestamps include timezone offsets (like 'Z')
configured_repos['createdAt'] = pd.to_datetime(configured_repos['createdAt'], utc=True)
configured_repos['scanned_at'] = pd.to_datetime(configured_repos['scanned_at'], utc=True)

# 2. Compute the difference (Age)
# This creates a Timedelta column
configured_repos['age_delta'] = configured_repos['scanned_at'] - configured_repos['createdAt']

# 3. Extract specific units
# Age in days (as integer)
configured_repos['repo_age_days'] = configured_repos['age_delta'].dt.days

In [10]:
context_df['repo_name'].nunique(), context_df.groupby('repo_name').size().sum()

(4463, np.int64(9470))

In [14]:
pr_df.drop_duplicates(subset=['id'], inplace=True)

In [20]:
pr_df['repo_id'].nunique()
# len(configured_repos)-pr_df['repo_id'].nunique()

2472

In [6]:
# print(configured_repos.iloc[0, 7])
pr_feat_df['state'].value_counts(normalize=True)

state
merged    0.7724
closed    0.2276
Name: proportion, dtype: float64

#### The projects with no agent-authored PRs

In [5]:
def generate_repo_stats_table(df: pd.DataFrame):
    """
    Generate a LaTeX table with repository statistics from GitHub PR data.
'', ''
    Args:
        df (pd.DataFrame): DataFrame containing GitHub PR data with repository metadata
        output_file (str): Path to output LaTeX file
    """
    # Step 1: Remove duplicate repositories (keep first occurrence)
    df_unique = df.sort_values("stargazers", ascending=0).drop_duplicates(
        subset=['repo'], keep='first').copy()

    # Count security PRs per repository
    # agentic_prs = df.groupby('repo').size()
    # df_unique['agentic_pr_count'] = df_unique['repo'].map(agentic_prs).fillna(0)

    # Step 3: Prepare the statistics table
    metrics = {
        'Age (days)': 'repo_age_days',
        'Stars': 'stargazers',
        'Commits': 'commits',
        'Agentic-PRs': 'agent_prs',
        'Total PRs': 'totalPullRequests',
        'Contributors': 'contributors',
        # 'Users': 'mentionableUsersCount'
    }

    # Calculate statistics for each metric
    stats_data = []
    for metric_name, col_name in metrics.items():
        values = df_unique[col_name]
        stats_data.append({
            'Metric': metric_name,
            'Min.': np.min(values),
            'Median': np.median(values),
            'Mean': np.mean(values),
            'Max.': np.max(values)
        })

    # Create stats DataFrame
    stats_df = pd.DataFrame(stats_data)
    stats_df['Max.'] = stats_df['Max.'].apply(lambda x: f"{x:,}")
    stats_df['Median'] = stats_df['Median'].apply(lambda x: f"{x:,.2f}")
    stats_df['Mean'] = stats_df['Mean'].apply(lambda x: f"{x:,.2f}")

    # Step 4: Generate LaTeX table
    latex_table = stats_df.to_latex(
        index=False,
        column_format='lrrrr',
        float_format="{:0.2f}".format,
        caption=f'Statistics of the {df['repo'].nunique():,} studied projects and {df['agent_prs'].sum():,} agentic PRs.',
        label='tab:repo-stats',
        position='t!'
    )
    latex_table = latex_table.replace(".00", "").replace(".0 ", " ")
    # Save to file
    # with open(output_file, 'w') as f:
    #     f.write(latex_table)

    return latex_table, df_unique

#### Stats of the project with at least 5 agent-authored and 5 after the creation of the first instruction file

In [9]:
COUNT_PR_THRESHOLD = 20

five_pr_mask = (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD)
repos_with_5_pr_bef_aft = pr_five_df.loc[
    five_pr_mask,
    'repo'
].unique()
# repos_with_5_pr_bef_aft

In [11]:
# len(repos_with_5_pr_bef_aft)

523

In [14]:
context_df[context_df['repo_name'].isin(repos_with_5_pr_bef_aft)].to_csv("../data/context_files_subset.csv", index=None)

In [14]:
configured_repos[configured_repos['repo'].isin(repos_with_5_pr_bef_aft)]['agent_prs'].sum()

np.float64(53153.0)

In [9]:
latex_output, repo_stat_df = generate_repo_stats_table(configured_repos[configured_repos['repo'].isin(repos_with_5_pr_bef_aft)])

In [10]:
print(latex_output)

\begin{table}[t!]
\caption{Statistics of the 188 studied projects and 18,008 agentic PRs.}
\label{tab:repo-stats}
\begin{tabular}{lrrrr}
\toprule
Metric & Min. & Median & Mean & Max. \\
\midrule
Age (days) & 571 & 1,921.50 & 2,195.14 & 6,357 \\
Stars & 10 & 2,303.50 & 10,122.20 & 145,711 \\
Commits & 375 & 4,376 & 8,902.71 & 142,284 \\
Agentic-PRs & 10 & 44 & 95.79 & 1,447 \\
Total PRs & 45 & 2,320.50 & 5,543.40 & 73,940 \\
Contributors & 5 & 87.50 & 144.38 & 463 \\
\bottomrule
\end{tabular}
\end{table}



### Inter-rater reliability score

In [17]:
claude_annotated_data_refined = pd.read_excel("../data/annotated_data_final.xlsx")

In [30]:
annotated_data_df = pd.read_csv("../data/annotated_acf_edits_final.csv")

# Example categorical ratings from two different raters
rater1 = annotated_data_df['sub_category_orig']
rater2 = annotated_data_df['sub_category_kosei']

# Compute Cohen's Kappa
kappa = cohen_kappa_score(rater1, rater2)

In [19]:
kappa

0.7607804804691907

In [32]:
annotated_data_df = annotated_data_df.merge(
    claude_annotated_data_refined[['repo', 'github_link', 'definition']],
    on=['repo', 'github_link'],
    how='left'
)

In [31]:
# annotated_data_df['Unnamed: 15'].value_counts()
annotated_data_df.drop(columns=['description'], inplace=True)

In [33]:
annotated_data_df.to_csv("../data/annotated_acf_edits_final.csv", index=False)

### The performance of TopicGPT

In [4]:
random_sample = pd.read_csv("../data/acf_random_sample.csv")

In [6]:
(random_sample['good_label']==1).sum()/len(random_sample)

np.float64(0.87)